# Lab 3 - SRNet MLA-Strict Pre-Dec2 (Modal)

- Goal: MLA-PDF-safe student (no exported Add), pre-dec2 bottleneck, optional FSRCNN distillation
- I/O: 256x256x3 -> 256x256x3
- Backend: Modal detached run


## 1. Setup


In [ ]:
from __future__ import annotations

import json
import math
import os
import random
import subprocess
import time
from collections import Counter
from contextlib import nullcontext
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import modal
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageOps
from torch.utils.data import DataLoader, Dataset

try:
    import onnx
except Exception:
    onnx = None

try:
    import onnxruntime as ort
except Exception:
    ort = None


PROJECT_ROOT = Path.cwd().resolve()
DATA_ROOT = PROJECT_ROOT / "Data"
RUN_DAY = os.environ.get("LAB3_SRNET_STARTED_DAY", time.strftime("%Y-%m-%d"))
RUN_NAME = os.environ.get("LAB3_SRNET_RUN_NAME", f"lab3_srnet_dwg4_v1_{time.strftime('%Y%m%d_%H%M%S')}")
MODEL_ID = os.environ.get("LAB3_SRNET_MODEL_ID", "srnet_mla_strict_predec2_v1")

MODAL_GPU = os.environ.get("LAB3_SRNET_MODAL_GPU", "L40S")
MODAL_TIMEOUT_MINUTES = int(os.environ.get("LAB3_SRNET_MODAL_TIMEOUT_MINUTES", "1440"))
MODAL_MAX_FUNCTION_SECONDS = 86400


def modal_function_timeout_seconds(minutes: int) -> int:
    return max(10, min(MODAL_MAX_FUNCTION_SECONDS, int(minutes) * 60))


POLL_INTERVAL_MINUTES = int(os.environ.get("LAB3_SRNET_POLL_INTERVAL_MINUTES", "15"))
MODAL_DATA_VOLUME = os.environ.get("LAB3_SRNET_MODAL_DATA_VOLUME", "lab3-data")
MODAL_RUNS_VOLUME = os.environ.get("LAB3_SRNET_MODAL_RUNS_VOLUME", "lab3-runs")
SYNC_DATA_TO_VOLUME = os.environ.get("LAB3_SRNET_SYNC_DATA", "true").lower() in {"1", "true", "yes"}
FORCE_DATA_SYNC = os.environ.get("LAB3_SRNET_FORCE_DATA_SYNC", "false").lower() in {"1", "true", "yes"}
RUN_MODAL_SUBMISSION = os.environ.get("LAB3_SRNET_RUN_MODAL", "true").lower() in {"1", "true", "yes"}

FSRCNN_REFERENCE_VAL_PSNR = 24.063410100069913
HINET_V6_REFERENCE_VAL_PSNR = 23.930102400346236
CURRENT_SRNET_PLATEAU_PSNR = 23.1359355579723
BASELINE_LATENCY_JSON = os.environ.get("LAB3_SRNET_BASELINE_LATENCY_JSON", "")

IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".bmp"}
RESAMPLING_BICUBIC = getattr(Image, "Resampling", Image).BICUBIC
SAFE_LEAF_MODULES = {"Conv2d", "ConvTranspose2d", "LeakyReLU"}
PDF_ALLOWED_ONNX_OPS = {"Conv", "ConvTranspose", "LeakyRelu"}
ALLOWED_ONNX_OPS = set(PDF_ALLOWED_ONNX_OPS)
FORBIDDEN_ONNX_OPS = {"Add", "Concat", "Clip", "Mul", "Sub", "Relu", "Resize", "MatMul", "Softmax", "Sigmoid", "Transpose", "Tanh"}
REMOTE_DATA_ROOT = Path("/mnt/lab3-data/Data")
REMOTE_RUNS_ROOT = Path("/mnt/lab3-runs")
REMOTE_PROJECT_ROOT = Path("/root/project")
L3_TRAIN_SPLITS = (
    ("LR_HR/L3_HR_train1", "L3_LR/L3_L3_train1", "L3_HR_train1"),
    ("LR_HR/L3_HR_train2", "L3_LR/L3_L3_train2", "L3_HR_train2"),
    ("LR_HR/L3_HR_train3", "L3_LR/L3_LR_train3", "L3_HR_train3"),
    ("LR_HR/L3_HR_train4", "L3_LR/L3_LR_train4", "L3_HR_train4"),
    ("LR_HR/L3_HR_train", "L3_LR/L3_LR_train", "L3_HR_train"),
)
L3_VAL_DIRS = ("L3_HR_valid", "L3_LR_valid")
REQUIRED_L3_LOCAL_PATHS = [
    DATA_ROOT / "LR_HR",
    DATA_ROOT / "L3_LR",
    DATA_ROOT / "L3_HR_valid",
    DATA_ROOT / "L3_LR_valid",
]
REQUIRED_L3_VOLUME_PATHS = [
    "/Data/LR_HR",
    "/Data/L3_LR",
    "/Data/L3_HR_valid",
    "/Data/L3_LR_valid",
]

DEFAULT_TEACHER_CHECKPOINT_CANDIDATES = [
    "runs/013420_2904_fsrcnn_residual_96_40_m8_modal_resume5750_to6000_lr300/checkpoints/best.pth",
    "runs/021424_2904_fsrcnn_residual_96_40_m8_modal_resume5750_to6000_lr300/checkpoints/best.pth",
]
DEFAULT_WARM_START_CHECKPOINT = (
    "runs/2026-05-01/lab3_srnet_dwg4_v1_recovery_l3_20260501_1651/checkpoints/best.pth"
)


@dataclass
class RunConfig:
    model_id: str
    run_day: str
    run_name: str
    seed: int = 1337
    train_patch_size: int = 256
    eval_size: int = 256
    batch_size: int = 8
    epochs: int = 460
    lr: float = 1e-4
    min_lr: float = 5e-6
    warmup_epochs: int = 5
    weight_decay: float = 5e-5
    grad_clip_norm: float = 0.5
    early_stop_patience: int = 35
    num_workers: int = 2
    use_amp: bool = True
    use_ema: bool = True
    ema_decay: float = 0.999
    calibration_count: int = 128
    warm_start_checkpoint: str = ""
    warm_start_selected_weight_source: str = "ema"
    teacher_checkpoint: str = ""
    teacher_weight_source: str = "ema"
    distill_weight: float = 0.05
    allow_missing_teacher: bool = True
    strict_mla_pdf_audit: bool = True
    modal_gpu: str = MODAL_GPU
    modal_timeout_minutes: int = MODAL_TIMEOUT_MINUTES
    poll_interval_minutes: int = POLL_INTERVAL_MINUTES
    modal_data_volume: str = MODAL_DATA_VOLUME
    modal_runs_volume: str = MODAL_RUNS_VOLUME


@dataclass
class RunPaths:
    run_day: str
    run_name: str
    run_root: Path
    checkpoints: Path
    exports: Path
    calibration: Path
    notebooks: Path
    config_json: Path
    metrics_jsonl: Path
    status_json: Path
    summary_json: Path
    report_json: Path
    best_checkpoint: Path
    last_checkpoint: Path
    best_onnx: Path
    calibration_manifest: Path
    conversion_script: Path
    latency_script: Path
    latency_json: Path
    mxq_target: Path


def build_paths(project_root: Path, run_day: str, run_name: str) -> RunPaths:
    run_root = project_root / "runs" / run_day / run_name
    checkpoints = run_root / "checkpoints"
    exports = run_root / "exports"
    calibration = exports / "calibration"
    notebooks = run_root / "notebooks"
    for current_path in [run_root, checkpoints, exports, calibration, notebooks]:
        current_path.mkdir(parents=True, exist_ok=True)
    return RunPaths(
        run_day=run_day,
        run_name=run_name,
        run_root=run_root,
        checkpoints=checkpoints,
        exports=exports,
        calibration=calibration,
        notebooks=notebooks,
        config_json=run_root / "run_config.json",
        metrics_jsonl=run_root / "metrics.jsonl",
        status_json=run_root / "latest_status.json",
        summary_json=run_root / "summary.json",
        report_json=run_root / "report.json",
        best_checkpoint=checkpoints / "best.pth",
        last_checkpoint=checkpoints / "last.pth",
        best_onnx=exports / "best.onnx",
        calibration_manifest=calibration / "manifest.json",
        conversion_script=exports / "convert_srnet_mxq.py",
        latency_script=exports / "measure_srnet_npu_latency.py",
        latency_json=exports / "npu_latency.json",
        mxq_target=exports / "best.mxq",
    )


def save_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")


def append_jsonl(path: Path, payload: dict[str, Any]) -> None:
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(payload, sort_keys=True) + "\n")


def print_setup_summary(cfg: RunConfig) -> None:
    print(
        f"[SETUP] model={cfg.model_id} run={cfg.run_name} "
        f"data_root={DATA_ROOT} modal_gpu={cfg.modal_gpu} timeout_min={cfg.modal_timeout_minutes}"
    )


def print_run_header(cfg: RunConfig, paths: RunPaths, train_pairs: list[tuple[Path, Path, str]], val_pairs: list[tuple[Path, Path, str]]) -> None:
    print(
        f"[RUN] model={cfg.model_id} run={paths.run_name} "
        f"train_pairs={len(train_pairs)} val_pairs={len(val_pairs)} "
        f"epochs={cfg.epochs} batch={cfg.batch_size}"
    )


def print_model_summary(contract: dict[str, Any], ops: dict[str, int]) -> None:
    print(
        f"[MODEL] params={contract['params']} "
        f"input={contract['input_shape']} output={contract['output_shape']} ops={ops}"
    )


def print_epoch(epoch: int, epochs: int, lr: float, train_loss: float, train_psnr: float, val_psnr: float, input_psnr: float, delta_psnr: float, sec: float) -> None:
    print(
        f"[EPOCH {epoch:03d}/{epochs:03d}] "
        f"lr={lr:.2e} train_loss={train_loss:.5f} train_psnr={train_psnr:.3f} "
        f"val_psnr={val_psnr:.3f} input_psnr={input_psnr:.3f} delta={delta_psnr:+.3f} time={sec:.1f}s"
    )


def print_final_summary(result: dict[str, Any]) -> None:
    vs = result["validation_summary"]
    gates = result["promotion_gates"]
    print(
        f"[FINAL] val_psnr={vs['val_psnr']:.4f} "
        f"delta_vs_fsrcnn={vs['delta_vs_fsrcnn_reference']:+.4f} "
        f"psnr_gate={gates['psnr_beats_fsrcnn']} "
        f"latency_gate={gates['latency_beats_baseline']} run_root={result['run_root']}"
    )


def run_modal_command(args: list[str]) -> subprocess.CompletedProcess[str]:
    return subprocess.run(args, check=False, capture_output=True, text=True, cwd=str(PROJECT_ROOT))


def ensure_volume(name: str) -> None:
    volume = modal.Volume.from_name(name, create_if_missing=True)
    volume.hydrate()


def volume_path_exists(volume_name: str, remote_path: str) -> bool:
    completed = run_modal_command(["modal", "volume", "ls", volume_name, remote_path])
    return completed.returncode == 0


def sync_data_volume(local_data_root: Path, volume_name: str, force: bool = False) -> None:
    ensure_volume(volume_name)
    missing_l3_paths = [remote_path for remote_path in REQUIRED_L3_VOLUME_PATHS if not volume_path_exists(volume_name, remote_path)]
    if not force and not missing_l3_paths:
        return
    if not SYNC_DATA_TO_VOLUME and missing_l3_paths:
        raise FileNotFoundError({"missing_l3_paths": missing_l3_paths, "hint": "Set LAB3_SRNET_SYNC_DATA=true or upload the L3 directories to lab3-data."})
    uploads = {
        "LR_HR": "/Data/LR_HR",
        "L3_LR": "/Data/L3_LR",
        "L3_HR_valid": "/Data/L3_HR_valid",
        "L3_LR_valid": "/Data/L3_LR_valid",
    }
    for local_name, remote_path in uploads.items():
        if not force and remote_path not in missing_l3_paths:
            continue
        local_path = local_data_root / local_name
        if not local_path.exists():
            raise FileNotFoundError(f"Local L3 data directory not found: {local_path}")
        completed = run_modal_command(["modal", "volume", "put", volume_name, str(local_path), remote_path, "--force"])
        if completed.returncode != 0:
            raise RuntimeError(completed.stderr.strip() or completed.stdout.strip() or f"Failed to sync {local_path}")
    for remote_path in REQUIRED_L3_VOLUME_PATHS:
        if not volume_path_exists(volume_name, remote_path):
            raise FileNotFoundError(f"Missing required L3 volume path after sync: {remote_path}")


def sync_run_from_volume(run_day: str, run_name: str, volume_name: str) -> Path:
    ensure_volume(volume_name)
    local_day_root = PROJECT_ROOT / "runs" / run_day
    local_day_root.mkdir(parents=True, exist_ok=True)
    remote_path = f"/runs/{run_day}/{run_name}"
    completed = run_modal_command(["modal", "volume", "get", volume_name, remote_path, str(local_day_root), "--force"])
    if completed.returncode != 0:
        raise RuntimeError(completed.stderr.strip() or completed.stdout.strip() or f"Failed to sync {remote_path}")
    return local_day_root / run_name


def localize_payload(payload: Any, local_run_root: Path, local_data_root: Path) -> Any:
    if isinstance(payload, dict):
        return {key: localize_payload(value, local_run_root, local_data_root) for key, value in payload.items()}
    if isinstance(payload, list):
        return [localize_payload(value, local_run_root, local_data_root) for value in payload]
    if isinstance(payload, str):
        replacements = {
            str(REMOTE_RUNS_ROOT / "runs" / local_run_root.parent.name / local_run_root.name): str(local_run_root),
            str(REMOTE_DATA_ROOT): str(local_data_root),
            str(REMOTE_PROJECT_ROOT): str(PROJECT_ROOT),
        }
        text = payload
        for old, new in replacements.items():
            text = text.replace(old, new)
        return text
    return payload


def normalize_synced_run(local_run_root: Path, local_data_root: Path) -> None:
    for name in ["run_config.json", "summary.json", "report.json", "latest_status.json"]:
        current_path = local_run_root / name
        if not current_path.exists():
            continue
        payload = json.loads(current_path.read_text(encoding="utf-8"))
        normalized = localize_payload(payload, local_run_root, local_data_root)
        save_json(current_path, normalized)


cfg = RunConfig(
    model_id=MODEL_ID,
    run_day=RUN_DAY,
    run_name=RUN_NAME,
    batch_size=int(os.environ.get("LAB3_SRNET_BATCH_SIZE", "8")),
    epochs=int(os.environ.get("LAB3_SRNET_EPOCHS", "460")),
    lr=float(os.environ.get("LAB3_SRNET_LR", "1e-4")),
    min_lr=float(os.environ.get("LAB3_SRNET_MIN_LR", "5e-6")),
    warmup_epochs=int(os.environ.get("LAB3_SRNET_WARMUP_EPOCHS", "5")),
    weight_decay=float(os.environ.get("LAB3_SRNET_WEIGHT_DECAY", "5e-5")),
    grad_clip_norm=float(os.environ.get("LAB3_SRNET_GRAD_CLIP_NORM", "0.5")),
    early_stop_patience=int(os.environ.get("LAB3_SRNET_EARLY_STOP_PATIENCE", "35")),
    warm_start_checkpoint=os.environ.get("LAB3_SRNET_WARM_START_CHECKPOINT", DEFAULT_WARM_START_CHECKPOINT),
    warm_start_selected_weight_source=os.environ.get("LAB3_SRNET_WARM_START_SELECTED_WEIGHT_SOURCE", "ema"),
    teacher_checkpoint=os.environ.get("LAB3_SRNET_TEACHER_CHECKPOINT", ""),
    teacher_weight_source=os.environ.get("LAB3_SRNET_TEACHER_WEIGHT_SOURCE", "ema"),
    distill_weight=float(os.environ.get("LAB3_SRNET_DISTILL_WEIGHT", "0.05")),
    allow_missing_teacher=os.environ.get("LAB3_SRNET_ALLOW_MISSING_TEACHER", "true").lower() in {"1", "true", "yes"},
    strict_mla_pdf_audit=os.environ.get("LAB3_SRNET_STRICT_MLA_PDF_AUDIT", "true").lower() in {"1", "true", "yes"},
)


## 2. Data


In [ ]:
def pil_rgb(path: Path) -> Image.Image:
    return Image.open(path).convert("RGB")


def pil_to_tensor(img: Image.Image) -> torch.Tensor:
    arr = np.asarray(img, dtype=np.float32) / 255.0
    arr = np.transpose(arr, (2, 0, 1))
    return torch.from_numpy(arr)


def collect_split_pairs(hr_dir: Path, lr_dir: Path, split_name: str) -> list[tuple[Path, Path, str]]:
    if not hr_dir.exists() or not lr_dir.exists():
        return []
    hr_images = {p.stem: p for p in hr_dir.iterdir() if p.suffix.lower() in IMAGE_SUFFIXES}
    lr_images = {p.stem: p for p in lr_dir.iterdir() if p.suffix.lower() in IMAGE_SUFFIXES}
    return [(lr_images[stem], hr_images[stem], f"{split_name}/{stem}") for stem in sorted(set(hr_images) & set(lr_images))]


def collect_train_pairs(data_root: Path) -> list[tuple[Path, Path, str]]:
    pairs: list[tuple[Path, Path, str]] = []
    for hr_rel, lr_rel, split_name in L3_TRAIN_SPLITS:
        pairs.extend(collect_split_pairs(data_root / hr_rel, data_root / lr_rel, split_name))
    return pairs


def collect_val_pairs(data_root: Path) -> list[tuple[Path, Path, str]]:
    hr_dir, lr_dir = L3_VAL_DIRS
    return collect_split_pairs(data_root / hr_dir, data_root / lr_dir, "val")


def random_crop_pair(lr_img: Image.Image, hr_img: Image.Image, size: int, rng: random.Random) -> tuple[Image.Image, Image.Image]:
    width, height = lr_img.size
    if min(width, height) < size:
        lr_img = ImageOps.fit(lr_img, (size, size), method=RESAMPLING_BICUBIC)
        hr_img = ImageOps.fit(hr_img, (size, size), method=RESAMPLING_BICUBIC)
        return lr_img, hr_img
    left = rng.randint(0, width - size)
    top = rng.randint(0, height - size)
    box = (left, top, left + size, top + size)
    return lr_img.crop(box), hr_img.crop(box)


def augment_pair(lr_img: Image.Image, hr_img: Image.Image, rng: random.Random) -> tuple[Image.Image, Image.Image]:
    if rng.random() < 0.5:
        lr_img = ImageOps.mirror(lr_img)
        hr_img = ImageOps.mirror(hr_img)
    if rng.random() < 0.5:
        lr_img = ImageOps.flip(lr_img)
        hr_img = ImageOps.flip(hr_img)
    rotations = rng.randint(0, 3)
    if rotations:
        angle = 90 * rotations
        lr_img = lr_img.rotate(angle)
        hr_img = hr_img.rotate(angle)
    return lr_img, hr_img


class PairedSRDataset(Dataset):
    def __init__(self, pairs: list[tuple[Path, Path, str]], train: bool, seed: int, patch_size: int, eval_size: int):
        self.pairs = pairs
        self.train = train
        self.seed = seed
        self.patch_size = patch_size
        self.eval_size = eval_size

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor, str]:
        lr_path, hr_path, name = self.pairs[index]
        lr_img = pil_rgb(lr_path)
        hr_img = pil_rgb(hr_path)
        rng = random.Random(self.seed + index)
        if self.train:
            lr_img, hr_img = random_crop_pair(lr_img, hr_img, self.patch_size, rng)
            lr_img, hr_img = augment_pair(lr_img, hr_img, rng)
        else:
            if lr_img.size != (self.eval_size, self.eval_size):
                lr_img = ImageOps.fit(lr_img, (self.eval_size, self.eval_size), method=RESAMPLING_BICUBIC)
            if hr_img.size != (self.eval_size, self.eval_size):
                hr_img = ImageOps.fit(hr_img, (self.eval_size, self.eval_size), method=RESAMPLING_BICUBIC)
        return pil_to_tensor(lr_img), pil_to_tensor(hr_img), name


def make_dataloaders(train_pairs: list[tuple[Path, Path, str]], val_pairs: list[tuple[Path, Path, str]], cfg: RunConfig, device: torch.device) -> tuple[DataLoader, DataLoader]:
    pin_memory = device.type == "cuda"
    train_loader = DataLoader(
        PairedSRDataset(train_pairs, train=True, seed=cfg.seed, patch_size=cfg.train_patch_size, eval_size=cfg.eval_size),
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=pin_memory,
        drop_last=False,
    )
    val_loader = DataLoader(
        PairedSRDataset(val_pairs, train=False, seed=cfg.seed, patch_size=cfg.train_patch_size, eval_size=cfg.eval_size),
        batch_size=max(1, min(cfg.batch_size, 8)),
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=pin_memory,
        drop_last=False,
    )
    return train_loader, val_loader


def tensor_psnr(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    mse = F.mse_loss(pred.clamp(0, 1), target.clamp(0, 1), reduction="none").mean(dim=(1, 2, 3)).clamp_min(1e-12)
    return -10.0 * torch.log10(mse)


## 3. Model and Training


In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def resolve_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def autocast_context(device: torch.device, use_amp: bool):
    if use_amp and device.type == "cuda":
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()


def move_batch(lr_img: torch.Tensor, hr_img: torch.Tensor, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    return lr_img.to(device, non_blocking=True), hr_img.to(device, non_blocking=True)


def init_tail_small(conv: nn.Conv2d, scale: float = 1e-3) -> None:
    nn.init.kaiming_normal_(conv.weight, a=0.1, mode="fan_in", nonlinearity="leaky_relu")
    conv.weight.data.mul_(scale)
    nn.init.zeros_(conv.bias)


class FSRCNNResidualNoUpscale(nn.Module):
    """FSRCNN-style residual refiner for 256x256 RGB (teacher only; not exported)."""

    def __init__(self, channels: int = 96, shrink_channels: int = 40, mapping_layers: int = 8, slope: float = 0.10):
        super().__init__()
        self.channels = channels
        self.shrink_channels = shrink_channels
        self.mapping_layers = mapping_layers
        self.slope = slope
        self.features = nn.Sequential(*self._feature_layers())
        self.tail = nn.Conv2d(channels, 3, 3, padding=1, bias=True)
        self.init_tail_small()

    def _activation(self) -> nn.Module:
        return nn.LeakyReLU(self.slope, inplace=False)

    def _feature_layers(self) -> list[nn.Module]:
        stem = [nn.Conv2d(3, self.channels, 5, padding=2, bias=True), self._activation()]
        shrink = [nn.Conv2d(self.channels, self.shrink_channels, 1, padding=0, bias=True), self._activation()]
        mapping: list[nn.Module] = []
        for _ in range(self.mapping_layers):
            mapping.extend([nn.Conv2d(self.shrink_channels, self.shrink_channels, 3, padding=1, bias=True), self._activation()])
        expand = [nn.Conv2d(self.shrink_channels, self.channels, 1, padding=0, bias=True), self._activation()]
        return [*stem, *shrink, *mapping, *expand]

    def init_tail_small(self) -> None:
        nn.init.kaiming_normal_(self.tail.weight, nonlinearity="linear")
        self.tail.weight.data.mul_(1e-3)
        if self.tail.bias is not None:
            nn.init.zeros_(self.tail.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.tail(self.features(x))


class ResidualCleanBlock(nn.Module):
    def __init__(self, channels: int, kernel_size: int):
        super().__init__()
        padding = kernel_size // 2
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size, padding=padding, bias=True),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(channels, channels, kernel_size, padding=padding, bias=True),
            nn.LeakyReLU(0.1, inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class DownsampleBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, stride=2, padding=1, bias=True),
            nn.LeakyReLU(0.1, inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class UpsampleBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.ConvTranspose2d(in_channels, out_channels, 4, stride=2, padding=1, bias=True),
            nn.LeakyReLU(0.1, inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class SRNetMLAStrictPreDec2Bottleneck(nn.Module):
    """MLA-safe student: no inference Add; direct image output; pre-dec2 128px bottleneck."""

    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, 32, 3, padding=1, bias=True), nn.LeakyReLU(0.1, inplace=True))
        self.enc1 = nn.Sequential(
            ResidualCleanBlock(32, 3),
            ResidualCleanBlock(32, 3),
            ResidualCleanBlock(32, 3),
        )
        self.down1 = DownsampleBlock(32, 64)
        self.enc2 = nn.Sequential(
            ResidualCleanBlock(64, 5),
            ResidualCleanBlock(64, 5),
            ResidualCleanBlock(64, 5),
        )
        self.down2 = DownsampleBlock(64, 128)
        self.bottleneck = nn.Sequential(
            ResidualCleanBlock(128, 3),
            ResidualCleanBlock(128, 5),
            ResidualCleanBlock(128, 3),
            ResidualCleanBlock(128, 5),
            ResidualCleanBlock(128, 3),
        )
        self.up1 = UpsampleBlock(128, 64)
        self.dec1 = nn.Sequential(
            ResidualCleanBlock(64, 5),
            ResidualCleanBlock(64, 5),
            ResidualCleanBlock(64, 5),
        )
        self.up2 = UpsampleBlock(64, 32)
        self.pre_dec2_down = DownsampleBlock(32, 48)
        self.pre_dec2_mid = nn.Sequential(
            ResidualCleanBlock(48, 3),
            ResidualCleanBlock(48, 5),
            ResidualCleanBlock(48, 3),
        )
        self.pre_dec2_up = UpsampleBlock(48, 32)
        self.tail_head = nn.Sequential(
            nn.Conv2d(32, 32, 3, padding=1, bias=True),
            nn.LeakyReLU(0.1, inplace=True),
        )
        self.tail = nn.Conv2d(32, 3, 3, padding=1, bias=True)
        init_tail_small(self.tail)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        stem = self.stem(x)
        enc1 = self.enc1(stem)
        enc2 = self.enc2(self.down1(enc1))
        bottleneck = self.bottleneck(self.down2(enc2))
        dec1 = self.dec1(self.up1(bottleneck))
        dec2_feat = self.up2(dec1)
        z = self.pre_dec2_up(self.pre_dec2_mid(self.pre_dec2_down(dec2_feat)))
        h = self.tail_head(z)
        return self.tail(h)


def operator_audit(model: nn.Module) -> dict[str, int]:
    counts: dict[str, int] = {}
    for module in model.modules():
        if list(module.children()):
            continue
        name = module.__class__.__name__
        counts[name] = counts.get(name, 0) + 1
    return dict(sorted(counts.items()))


def validate_operator_audit(model: nn.Module) -> None:
    bad = []
    for name, module in model.named_modules():
        if name and not list(module.children()) and module.__class__.__name__ not in SAFE_LEAF_MODULES:
            bad.append(f"{name}: {module.__class__.__name__}")
    if bad:
        raise RuntimeError(f"Unexpected leaf modules: {bad}")


def verify_model_contract(model: nn.Module, eval_size: int) -> dict[str, Any]:
    validate_operator_audit(model)
    model_device = next(model.parameters()).device
    dummy = torch.zeros(1, 3, eval_size, eval_size, device=model_device)
    with torch.no_grad():
        out = model(dummy)
    if tuple(out.shape) != (1, 3, eval_size, eval_size):
        raise RuntimeError(f"Expected 1x3x{eval_size}x{eval_size}, got {tuple(out.shape)}")
    return {
        "input_shape": list(dummy.shape),
        "output_shape": list(out.shape),
        "params": sum(p.numel() for p in model.parameters()),
    }


def compute_supervised_loss(pred: torch.Tensor, lr: torch.Tensor, hr: torch.Tensor) -> torch.Tensor:
    pred_res = pred - lr
    target_res = hr - lr
    loss_mse = F.mse_loss(pred_res, target_res)
    loss_l1 = F.l1_loss(pred_res, target_res)
    return 0.8 * loss_mse + 0.2 * loss_l1


def compute_train_loss(
    pred: torch.Tensor,
    lr: torch.Tensor,
    hr: torch.Tensor,
    teacher: nn.Module | None,
    cfg: RunConfig,
) -> torch.Tensor:
    loss_sup = compute_supervised_loss(pred, lr, hr)
    if teacher is None or cfg.distill_weight <= 0.0:
        return loss_sup
    with torch.no_grad():
        teacher_pred = teacher(lr)
    pred_res = pred - lr
    teacher_res = teacher_pred - lr
    loss_distill = F.mse_loss(pred_res, teacher_res)
    return loss_sup + cfg.distill_weight * loss_distill


def create_ema_model(model: nn.Module) -> nn.Module:
    ema_model = SRNetMLAStrictPreDec2Bottleneck()
    ema_model.load_state_dict(model.state_dict())
    ema_model.requires_grad_(False)
    ema_model.eval()
    return ema_model


@torch.no_grad()
def ema_update(ema_model: nn.Module, model: nn.Module, decay: float) -> None:
    for ema_param, param in zip(ema_model.parameters(), model.parameters()):
        ema_param.data.mul_(decay).add_(param.data, alpha=1.0 - decay)


def checkpoint_payload(
    model: nn.Module,
    ema_model: nn.Module | None,
    optimizer: torch.optim.Optimizer,
    epoch: int,
    best_val_psnr: float,
    latest_metrics: dict[str, Any],
    cfg: RunConfig,
    warm_start: dict[str, Any],
    stop_reason: str,
) -> dict[str, Any]:
    return {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "ema_model_state_dict": ema_model.state_dict() if ema_model is not None else None,
        "optimizer_state_dict": optimizer.state_dict(),
        "best_val_psnr": best_val_psnr,
        "latest_metrics": latest_metrics,
        "config": asdict(cfg),
        "selected_weight_source": "ema" if ema_model is not None else "model",
        "warm_start": warm_start,
        "stop_reason": stop_reason,
    }


def save_checkpoint(
    path: Path,
    model: nn.Module,
    ema_model: nn.Module | None,
    optimizer: torch.optim.Optimizer,
    epoch: int,
    best_val_psnr: float,
    latest_metrics: dict[str, Any],
    cfg: RunConfig,
    warm_start: dict[str, Any],
    stop_reason: str,
) -> None:
    torch.save(checkpoint_payload(model, ema_model, optimizer, epoch, best_val_psnr, latest_metrics, cfg, warm_start, stop_reason), path)


def select_checkpoint_state(checkpoint: dict[str, Any], preferred_source: str) -> dict[str, torch.Tensor]:
    preferred = preferred_source.lower().strip()
    if preferred in {"ema", "ema_model_state_dict"}:
        if checkpoint.get("ema_model_state_dict") is not None:
            return checkpoint["ema_model_state_dict"]
        if checkpoint.get("ema_state_dict") is not None:
            return checkpoint["ema_state_dict"]
    if checkpoint.get("model_state_dict") is not None:
        return checkpoint["model_state_dict"]
    raise KeyError("Checkpoint does not contain a supported model state dict.")


def load_model_from_checkpoint(path: Path, device: torch.device) -> nn.Module:
    checkpoint = torch.load(path, map_location=device)
    preferred_source = checkpoint.get("selected_weight_source", "ema")
    state_dict = select_checkpoint_state(checkpoint, preferred_source)
    model = SRNetMLAStrictPreDec2Bottleneck().to(device)
    model.load_state_dict(state_dict, strict=True)
    model.eval()
    return model


def resolve_checkpoint_path(checkpoint_value: str, checkpoint_base: Path) -> Path | None:
    if not checkpoint_value:
        return None
    candidate = Path(checkpoint_value)
    return candidate if candidate.is_absolute() else checkpoint_base / candidate


def maybe_load_warm_start(model: nn.Module, cfg: RunConfig, device: torch.device, checkpoint_base: Path) -> dict[str, Any]:
    skipped_keys: list[str] = []
    info: dict[str, Any] = {
        "status": "disabled",
        "path": None,
        "source": cfg.warm_start_selected_weight_source,
        "loaded_keys": [],
        "loaded_key_count": 0,
        "skipped_key_count": 0,
        "skipped_keys": [],
        "rejected_reason": None,
    }
    checkpoint_path = resolve_checkpoint_path(cfg.warm_start_checkpoint, checkpoint_base)
    if checkpoint_path is None:
        return info
    info["path"] = str(checkpoint_path)
    if not checkpoint_path.exists():
        info["status"] = "rejected"
        info["rejected_reason"] = "checkpoint_not_found"
        return info
    checkpoint = torch.load(checkpoint_path, map_location=device)
    try:
        source_state = select_checkpoint_state(checkpoint, cfg.warm_start_selected_weight_source)
    except KeyError:
        info["status"] = "rejected"
        info["rejected_reason"] = "selected_source_missing"
        return info
    target_state = model.state_dict()
    mismatched = sorted(
        key for key, value in source_state.items() if key in target_state and target_state[key].shape != value.shape
    )
    if mismatched:
        info["status"] = "rejected"
        info["rejected_reason"] = f"shape_mismatch:{','.join(mismatched[:8])}"
        return info
    compatible = {key: value for key, value in source_state.items() if key in target_state and target_state[key].shape == value.shape}
    if not compatible:
        info["status"] = "rejected"
        info["rejected_reason"] = "no_compatible_keys"
        return info
    model.load_state_dict(compatible, strict=False)
    skipped_keys = sorted(key for key in source_state if key not in compatible)
    info["status"] = "loaded"
    info["loaded_keys"] = sorted(compatible)
    info["loaded_key_count"] = len(compatible)
    info["skipped_keys"] = skipped_keys
    info["skipped_key_count"] = len(skipped_keys)
    return info


def resolve_teacher_model(cfg: RunConfig, device: torch.device, checkpoint_base: Path) -> tuple[nn.Module | None, dict[str, Any]]:
    info: dict[str, Any] = {
        "enabled": False,
        "path": None,
        "source": cfg.teacher_weight_source,
        "resolved_automatically": False,
        "rejected_reason": None,
    }
    candidates: list[str] = []
    if cfg.teacher_checkpoint.strip():
        candidates.append(cfg.teacher_checkpoint.strip())
    else:
        candidates.extend(DEFAULT_TEACHER_CHECKPOINT_CANDIDATES)
    for rel in candidates:
        path = resolve_checkpoint_path(rel, checkpoint_base)
        if path is None or not path.exists():
            continue
        try:
            ckpt = torch.load(path, map_location=device)
            state = select_checkpoint_state(ckpt, cfg.teacher_weight_source)
        except KeyError:
            info["rejected_reason"] = "selected_source_missing"
            continue
        except Exception as exc:
            info["rejected_reason"] = f"load_failed:{type(exc).__name__}"
            continue
        teach = FSRCNNResidualNoUpscale().to(device)
        try:
            teach.load_state_dict(state, strict=True)
        except Exception as exc:
            info["rejected_reason"] = f"state_dict_mismatch:{type(exc).__name__}"
            continue
        teach.eval()
        teach.requires_grad_(False)
        info["enabled"] = True
        info["path"] = str(path)
        info["resolved_automatically"] = not bool(cfg.teacher_checkpoint.strip())
        return teach, info
    if cfg.allow_missing_teacher:
        info["rejected_reason"] = info["rejected_reason"] or "no_teacher_checkpoint_found"
        return None, info
    raise FileNotFoundError({"teacher_candidates": candidates, "checkpoint_base": str(checkpoint_base)})


def train_one_epoch(
    model: nn.Module,
    ema_model: nn.Module | None,
    teacher: nn.Module | None,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scaler: Any,
    device: torch.device,
    cfg: RunConfig,
) -> dict[str, float | int]:
    model.train()
    total_loss = 0.0
    total_psnr = 0.0
    sample_count = 0
    nonfinite_step_count = 0
    for lr_img, hr_img, _ in loader:
        lr_img, hr_img = move_batch(lr_img, hr_img, device)
        optimizer.zero_grad(set_to_none=True)
        with autocast_context(device, cfg.use_amp):
            pred = model(lr_img)
            loss = compute_train_loss(pred, lr_img, hr_img, teacher, cfg)
        if not torch.isfinite(loss).all():
            nonfinite_step_count += 1
            continue
        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip_norm)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip_norm)
            optimizer.step()
        if ema_model is not None:
            ema_update(ema_model, model, cfg.ema_decay)
        batch_size = lr_img.size(0)
        total_loss += float(loss.item()) * batch_size
        with torch.no_grad():
            total_psnr += tensor_psnr(pred.detach(), hr_img).sum().item()
        sample_count += batch_size
    train_loss = float("nan") if sample_count == 0 else total_loss / sample_count
    train_psnr = float("nan") if sample_count == 0 else total_psnr / sample_count
    return {
        "train_loss": train_loss,
        "train_psnr": train_psnr,
        "nonfinite_step_count": nonfinite_step_count,
    }


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device: torch.device, cfg: RunConfig) -> dict[str, float]:
    model.eval()
    total_loss = 0.0
    total_psnr = 0.0
    total_input_psnr = 0.0
    total_delta = 0.0
    sample_count = 0
    for lr_img, hr_img, _ in loader:
        lr_img, hr_img = move_batch(lr_img, hr_img, device)
        with autocast_context(device, cfg.use_amp):
            pred = model(lr_img)
            loss = compute_supervised_loss(pred, lr_img, hr_img)
        pred_psnr = tensor_psnr(pred, hr_img)
        input_psnr = tensor_psnr(lr_img, hr_img)
        delta = pred_psnr - input_psnr
        batch_size = lr_img.size(0)
        total_loss += float(loss.item()) * batch_size
        total_psnr += pred_psnr.sum().item()
        total_input_psnr += input_psnr.sum().item()
        total_delta += delta.sum().item()
        sample_count += batch_size
    return {
        "val_loss": total_loss / max(1, sample_count),
        "val_psnr": total_psnr / max(1, sample_count),
        "input_psnr": total_input_psnr / max(1, sample_count),
        "delta_psnr": total_delta / max(1, sample_count),
    }


def lr_multiplier(epoch: int, total_epochs: int, warmup_epochs: int, min_lr_ratio: float) -> float:
    if epoch < warmup_epochs:
        return float(epoch + 1) / float(max(1, warmup_epochs))
    progress = (epoch - warmup_epochs) / float(max(1, total_epochs - warmup_epochs))
    cosine = 0.5 * (1.0 + math.cos(math.pi * min(max(progress, 0.0), 1.0)))
    return min_lr_ratio + (1.0 - min_lr_ratio) * cosine


def audit_onnx_graph(onnx_path: Path, strict_pdf: bool) -> dict[str, Any]:
    if onnx is None:
        return {"checked": False, "reason": "onnx unavailable"}
    loaded = onnx.load(str(onnx_path))
    onnx.checker.check_model(loaded)
    ops = [node.op_type for node in loaded.graph.node]
    counts = dict(sorted(Counter(ops).items()))
    forbidden = {op: counts[op] for op in sorted(FORBIDDEN_ONNX_OPS) if op in counts}
    unexpected = {op: counts[op] for op in sorted(set(counts) - ALLOWED_ONNX_OPS)}
    pdf_bad = {op: counts[op] for op in sorted(set(counts)) if op not in PDF_ALLOWED_ONNX_OPS}
    pdf_allowed_ops_only = len(pdf_bad) == 0
    payload: dict[str, Any] = {
        "checked": True,
        "onnx_checker": "passed",
        "op_counts": counts,
        "forbidden_ops_found": forbidden,
        "unexpected_ops_found": unexpected,
        "allowed_ops_only": not unexpected and not forbidden,
        "forbidden_ops_absent": not forbidden,
        "pdf_allowed_ops_only": pdf_allowed_ops_only,
        "pdf_unexpected_ops_found": pdf_bad,
        "strict_mla_pdf_audit": strict_pdf,
    }
    return payload


def export_to_onnx(model: nn.Module, onnx_path: Path, device: torch.device, eval_size: int, cfg: RunConfig) -> dict[str, Any]:
    model.eval()
    dummy = torch.zeros(1, 3, eval_size, eval_size, device=device)
    with torch.no_grad():
        torch.onnx.export(
            model,
            dummy,
            str(onnx_path),
            export_params=True,
            opset_version=13,
            do_constant_folding=True,
            input_names=["input"],
            output_names=["output"],
            dynamic_axes=None,
            dynamo=False,
        )
    result: dict[str, Any] = {"onnx_path": str(onnx_path)}
    graph_audit = audit_onnx_graph(onnx_path, cfg.strict_mla_pdf_audit)
    result.update(graph_audit)
    strict_fail = graph_audit.get("checked") and cfg.strict_mla_pdf_audit and (
        not graph_audit.get("pdf_allowed_ops_only", False)
        or not graph_audit.get("forbidden_ops_absent", True)
    )
    legacy_fail = graph_audit.get("checked") and not cfg.strict_mla_pdf_audit and (
        not graph_audit.get("forbidden_ops_absent", True) or not graph_audit.get("allowed_ops_only", True)
    )
    if strict_fail or legacy_fail:
        raise RuntimeError({"onnx_path": str(onnx_path), **graph_audit})
    if ort is not None:
        cpu_model = SRNetMLAStrictPreDec2Bottleneck().cpu().eval()
        cpu_model.load_state_dict(model.state_dict(), strict=True)
        parity_input = torch.zeros(1, 3, eval_size, eval_size)
        with torch.no_grad():
            torch_output = cpu_model(parity_input).detach().cpu().numpy()
        session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
        ort_output = session.run(["output"], {"input": parity_input.numpy()})[0]
        diff = np.abs(ort_output - torch_output)
        result["ort_max_diff"] = float(diff.max())
        result["ort_mean_diff"] = float(diff.mean())
        result["parity_pass"] = bool(float(diff.max()) < 1e-3)
        if not result["parity_pass"]:
            raise RuntimeError({"onnx_path": str(onnx_path), "ort_max_diff": result["ort_max_diff"], "ort_mean_diff": result["ort_mean_diff"]})
    return result


def write_calibration_set(train_pairs: list[tuple[Path, Path, str]], calibration_dir: Path, count: int, eval_size: int) -> dict[str, Any]:
    calibration_dir.mkdir(parents=True, exist_ok=True)
    items = []
    for idx, (lr_path, _hr_path, name) in enumerate(train_pairs[:count]):
        dst = calibration_dir / f"{idx:04d}_{Path(name).stem}.png"
        pil_rgb(lr_path).resize((eval_size, eval_size), RESAMPLING_BICUBIC).save(dst)
        items.append({
            "source_lr": str(lr_path),
            "calibration_image": str(dst),
            "derived_from_training": True,
        })
    payload = {"count": len(items), "derived_from_training": True, "items": items}
    save_json(calibration_dir / "manifest.json", payload)
    return payload


def write_conversion_script(path: Path) -> None:
    script = """#!/usr/bin/env python3
from __future__ import annotations
import argparse
from pathlib import Path

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--onnx', required=True)
    parser.add_argument('--calibration-dir', required=True)
    parser.add_argument('--output', required=True)
    parser.add_argument('--dry-run', action='store_true')
    args = parser.parse_args()

    onnx_path = Path(args.onnx)
    calibration_dir = Path(args.calibration_dir)
    output_path = Path(args.output)
    if not onnx_path.exists():
        raise FileNotFoundError(onnx_path)
    if not (calibration_dir / 'manifest.json').exists():
        raise FileNotFoundError(calibration_dir / 'manifest.json')

    command = [
        'mobilint-converter',
        '--model', str(onnx_path),
        '--calibration', str(calibration_dir),
        '--output', str(output_path),
    ]
    print({'command': command, 'dry_run': args.dry_run})
    if args.dry_run:
        return
    raise SystemExit('Replace mobilint-converter invocation with the actual MLA host command.')

if __name__ == '__main__':
    main()
"""
    path.write_text(script, encoding="utf-8")
    path.chmod(0o755)


def write_latency_script(path: Path, model_id: str) -> None:
    script = f"""#!/usr/bin/env python3
from __future__ import annotations
import argparse
import json
import statistics
import time
from pathlib import Path

MODEL_ID = {model_id!r}

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--mxq', required=True)
    parser.add_argument('--output', required=True)
    parser.add_argument('--warmup', type=int, default=10)
    parser.add_argument('--iters', type=int, default=100)
    args = parser.parse_args()

    mxq_path = Path(args.mxq)
    out_path = Path(args.output)
    if not mxq_path.exists():
        raise FileNotFoundError(mxq_path)

    latencies_ms = []
    for _ in range(args.warmup):
        pass
    for _ in range(args.iters):
        start = time.perf_counter()
        # npu_model.run(...)
        end = time.perf_counter()
        latencies_ms.append((end - start) * 1000.0)

    payload = {{
        'model_id': MODEL_ID,
        'mxq_path': str(mxq_path),
        'iterations': args.iters,
        'mean_latency_ms': statistics.mean(latencies_ms) if latencies_ms else None,
        'p50_ms': statistics.median(latencies_ms) if latencies_ms else None,
        'min_ms': min(latencies_ms) if latencies_ms else None,
        'max_ms': max(latencies_ms) if latencies_ms else None,
    }}
    out_path.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    print(json.dumps(payload, indent=2))

if __name__ == '__main__':
    main()
"""
    path.write_text(script, encoding="utf-8")
    path.chmod(0o755)


def build_validation_summary(val_metrics: dict[str, float]) -> dict[str, Any]:
    return {
        "val_psnr": float(val_metrics["val_psnr"]),
        "input_psnr": float(val_metrics["input_psnr"]),
        "delta_psnr": float(val_metrics["delta_psnr"]),
        "beats_fsrcnn_reference": bool(val_metrics["val_psnr"] > FSRCNN_REFERENCE_VAL_PSNR),
        "delta_vs_fsrcnn_reference": float(val_metrics["val_psnr"] - FSRCNN_REFERENCE_VAL_PSNR),
    }


def run_pipeline(config_payload: dict[str, Any], data_root: Path, artifact_root: Path) -> dict[str, Any]:
    cfg = RunConfig(**config_payload)
    checkpoint_base = artifact_root
    paths = build_paths(artifact_root, cfg.run_day, cfg.run_name)
    save_json(paths.config_json, {**asdict(cfg), "data_root": str(data_root), "artifact_root": str(artifact_root)})

    set_seed(cfg.seed)
    device = resolve_device()
    train_pairs = collect_train_pairs(data_root)
    val_pairs = collect_val_pairs(data_root)
    if not train_pairs:
        raise FileNotFoundError(f"No training pairs found under {data_root}")
    if not val_pairs:
        raise FileNotFoundError(f"No validation pairs found under {data_root}")

    train_loader, val_loader = make_dataloaders(train_pairs, val_pairs, cfg, device)
    model = SRNetMLAStrictPreDec2Bottleneck().to(device)
    warm_start = maybe_load_warm_start(model, cfg, device, checkpoint_base)
    teacher_model, teacher_distillation = resolve_teacher_model(cfg, device, checkpoint_base)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    min_lr_ratio = cfg.min_lr / cfg.lr
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda=lambda epoch: lr_multiplier(epoch, cfg.epochs, cfg.warmup_epochs, min_lr_ratio),
    )
    scaler = torch.amp.GradScaler("cuda", enabled=cfg.use_amp and device.type == "cuda") if hasattr(torch, "amp") else None
    ema_model = create_ema_model(model).to(device) if cfg.use_ema else None

    best_val_psnr = float("-inf")
    best_epoch = 0
    epochs_without_improvement = 0
    total_nonfinite_step_count = 0
    latest_metrics: dict[str, Any] = {}
    stop_reason = "completed_all_epochs"

    for epoch in range(cfg.epochs):
        epoch_index = epoch + 1
        start_time = time.time()
        train_metrics = train_one_epoch(model, ema_model, teacher_model, train_loader, optimizer, scaler, device, cfg)
        epoch_nonfinite = int(train_metrics.pop("nonfinite_step_count"))
        total_nonfinite_step_count += epoch_nonfinite
        eval_model = ema_model if ema_model is not None else model
        val_metrics = evaluate(eval_model, val_loader, device, cfg)
        scheduler.step()
        latest_metrics = {
            "epoch": epoch_index,
            "learning_rate": float(optimizer.param_groups[0]["lr"]),
            "epoch_seconds": time.time() - start_time,
            "epoch_nonfinite_step_count": epoch_nonfinite,
            **train_metrics,
            **val_metrics,
        }
        is_best = latest_metrics["val_psnr"] > best_val_psnr
        if is_best:
            best_val_psnr = float(latest_metrics["val_psnr"])
            best_epoch = epoch_index
            epochs_without_improvement = 0
            save_checkpoint(paths.best_checkpoint, model, ema_model, optimizer, epoch_index, best_val_psnr, latest_metrics, cfg, warm_start, stop_reason)
        else:
            epochs_without_improvement += 1
        append_jsonl(paths.metrics_jsonl, latest_metrics)
        save_json(
            paths.status_json,
            {
                "phase": "training",
                "epoch": epoch_index,
                "best_epoch": best_epoch,
                "best_val_psnr": best_val_psnr,
                "latest_metrics": latest_metrics,
                "warm_start": warm_start,
                "teacher_distillation": teacher_distillation,
                "training_summary": {
                    "nonfinite_step_count": total_nonfinite_step_count,
                    "stop_reason": "training",
                },
            },
        )
        save_checkpoint(paths.last_checkpoint, model, ema_model, optimizer, epoch_index, best_val_psnr, latest_metrics, cfg, warm_start, stop_reason)
        print_epoch(
            epoch_index,
            cfg.epochs,
            latest_metrics["learning_rate"],
            latest_metrics["train_loss"],
            latest_metrics["train_psnr"],
            latest_metrics["val_psnr"],
            latest_metrics["input_psnr"],
            latest_metrics["delta_psnr"],
            latest_metrics["epoch_seconds"],
        )
        if epochs_without_improvement >= cfg.early_stop_patience:
            stop_reason = f"early_stop_no_val_improvement_{cfg.early_stop_patience}"
            save_checkpoint(paths.last_checkpoint, model, ema_model, optimizer, epoch_index, best_val_psnr, latest_metrics, cfg, warm_start, stop_reason)
            break

    eval_model = load_model_from_checkpoint(paths.best_checkpoint, device)
    val_metrics = evaluate(eval_model, val_loader, device, cfg)
    validation_summary = build_validation_summary(val_metrics)
    onnx_summary = export_to_onnx(eval_model, paths.best_onnx, device, cfg.eval_size, cfg)
    calibration_summary = write_calibration_set(train_pairs, paths.calibration, cfg.calibration_count, cfg.eval_size)
    write_conversion_script(paths.conversion_script)
    write_latency_script(paths.latency_script, cfg.model_id)

    training_summary = {
        "nonfinite_step_count": total_nonfinite_step_count,
        "stop_reason": stop_reason,
    }
    summary_payload = {
        "model_id": cfg.model_id,
        "run_root": str(paths.run_root),
        "best_epoch": best_epoch,
        "best_val_psnr": best_val_psnr,
        "latest_metrics": latest_metrics,
        "validation_summary": validation_summary,
        "onnx_summary": onnx_summary,
        "calibration_summary": calibration_summary,
        "warm_start": warm_start,
        "teacher_distillation": teacher_distillation,
        "training_summary": training_summary,
    }
    save_json(paths.summary_json, summary_payload)

    onnx_pass = bool(
        onnx_summary.get("parity_pass", True)
        and onnx_summary.get("forbidden_ops_absent", True)
        and (onnx_summary.get("pdf_allowed_ops_only", True) if cfg.strict_mla_pdf_audit else onnx_summary.get("allowed_ops_only", True))
    )

    final_result = {
        "model_id": cfg.model_id,
        "run_root": str(paths.run_root),
        "operator_audit": operator_audit(eval_model),
        "validation_summary": validation_summary,
        "onnx_summary": onnx_summary,
        "warm_start": warm_start,
        "teacher_distillation": teacher_distillation,
        "training_summary": training_summary,
        "artifact_paths": {
            "best_checkpoint": str(paths.best_checkpoint),
            "onnx": str(paths.best_onnx),
            "calibration_manifest": str(paths.calibration_manifest),
            "conversion_script": str(paths.conversion_script),
            "mxq_target": str(paths.mxq_target),
            "latency_script": str(paths.latency_script),
            "latency_json": str(paths.latency_json),
        },
        "promotion_gates": {
            "contract_pass": True,
            "safe_ops_pass": True,
            "onnx_pass": onnx_pass,
            "calibration_pass": True,
            "mxq_handoff_pass": paths.conversion_script.exists() and paths.latency_script.exists(),
            "psnr_beats_fsrcnn": validation_summary["beats_fsrcnn_reference"],
            "latency_beats_baseline": False,
            "promotion_pass": False,
        },
    }
    save_json(paths.report_json, final_result)
    save_json(
        paths.status_json,
        {
            "phase": "completed",
            "epoch": latest_metrics.get("epoch", 0),
            "best_epoch": best_epoch,
            "best_val_psnr": best_val_psnr,
            "latest_metrics": latest_metrics,
            "warm_start": warm_start,
            "teacher_distillation": teacher_distillation,
            "training_summary": training_summary,
        },
    )
    return final_result


NOTEBOOK_PATH = PROJECT_ROOT / "experiments" / "SRNet NPU v1" / "lab3_srnet_npu_v1_modal.ipynb"
REMOTE_RUNTIME_MODULE_PATH = "/root/srnet_modal_entry.py"


def materialize_runtime_module() -> Path:
    runtime_path = PROJECT_ROOT / "tmp" / "lab3_srnet_npu_v1_modal_runtime.py"
    runtime_path.parent.mkdir(parents=True, exist_ok=True)
    notebook = json.loads(NOTEBOOK_PATH.read_text(encoding="utf-8"))
    runtime_cells = []
    for cell in notebook["cells"]:
        if cell.get("cell_type") != "code":
            continue
        source = "".join(cell.get("source", []))
        first_line = source.lstrip().splitlines()[0] if source.strip() else ""
        if first_line.startswith("paths = build_paths("):
            continue
        if first_line.startswith("if not RUN_MODAL_SUBMISSION:"):
            continue
        runtime_cells.append(source)
    runtime_path.write_text("\n\n".join(runtime_cells) + "\n", encoding="utf-8")
    return runtime_path


def execute_modal_pipeline(cfg: RunConfig) -> dict[str, Any]:
    print(f"[MODAL] run_name={cfg.run_name} day={cfg.run_day} timeout_s={modal_function_timeout_seconds(cfg.modal_timeout_minutes)}", flush=True)
    if SYNC_DATA_TO_VOLUME or FORCE_DATA_SYNC or not volume_path_exists(cfg.modal_data_volume, "/Data"):
        print("[MODAL] sync_data_volume... (may take a while)", flush=True)
        sync_data_volume(DATA_ROOT, cfg.modal_data_volume, force=FORCE_DATA_SYNC)
        print("[MODAL] sync_data_volume done", flush=True)
    else:
        ensure_volume(cfg.modal_data_volume)
    ensure_volume(cfg.modal_runs_volume)

    print("[MODAL] materialize_runtime_module...", flush=True)
    runtime_module_path = materialize_runtime_module()
    print(f"[MODAL] runtime module {runtime_module_path}", flush=True)
    app = modal.App("lab3-srnet-modal")
    image = modal.Image.debian_slim(python_version="3.13").pip_install(
        "torch==2.10.0",
        "onnx==1.20.1",
        "onnxruntime==1.24.1",
        "numpy==2.4.2",
        "pillow==12.1.1",
    ).add_local_file(str(runtime_module_path), remote_path=REMOTE_RUNTIME_MODULE_PATH)
    data_volume = modal.Volume.from_name(cfg.modal_data_volume, create_if_missing=True)
    runs_volume = modal.Volume.from_name(cfg.modal_runs_volume, create_if_missing=True)

    print("[MODAL] app.run: spawn training (poll prints every poll_interval_minutes)...", flush=True)

    @app.function(
        image=image,
        gpu=cfg.modal_gpu,
        timeout=modal_function_timeout_seconds(cfg.modal_timeout_minutes),
        serialized=True,
        volumes={"/mnt/lab3-data": data_volume, "/mnt/lab3-runs": runs_volume},
    )
    def run_srnet_remote(config_payload: dict[str, Any]) -> dict[str, Any]:
        import importlib.util
        import sys

        spec = importlib.util.spec_from_file_location("srnet_modal_entry", REMOTE_RUNTIME_MODULE_PATH)
        module = importlib.util.module_from_spec(spec)
        assert spec.loader is not None
        sys.modules[spec.name] = module
        spec.loader.exec_module(module)
        result = module.run_pipeline(config_payload, module.REMOTE_DATA_ROOT, module.REMOTE_RUNS_ROOT)
        runs_volume.commit()
        return result

    with app.run():
        call = run_srnet_remote.spawn(asdict(cfg))
        while True:
            try:
                summary = call.get(timeout=max(60, cfg.poll_interval_minutes * 60))
                break
            except TimeoutError:
                print({"status": "still_running", "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")}, flush=True)

    print("[MODAL] remote finished; syncing runs volume to local...", flush=True)
    local_run_root = sync_run_from_volume(cfg.run_day, cfg.run_name, cfg.modal_runs_volume)
    normalize_synced_run(local_run_root, DATA_ROOT)
    report_path = local_run_root / "report.json"
    if report_path.exists():
        return json.loads(report_path.read_text(encoding="utf-8"))
    return localize_payload(summary, local_run_root, DATA_ROOT)


## 4. Validation


In [ ]:
paths = build_paths(PROJECT_ROOT, cfg.run_day, cfg.run_name)
set_seed(cfg.seed)
local_device = resolve_device()
missing_local_paths = [str(path) for path in REQUIRED_L3_LOCAL_PATHS if not path.exists()]
if missing_local_paths:
    raise FileNotFoundError({"missing_l3_local_paths": missing_local_paths})
train_pairs = collect_train_pairs(DATA_ROOT)
val_pairs = collect_val_pairs(DATA_ROOT)
if not train_pairs:
    raise FileNotFoundError(f"No L3 training pairs found under {DATA_ROOT}")
if not val_pairs:
    raise FileNotFoundError(f"No L3 validation pairs found under {DATA_ROOT}")
model = SRNetMLAStrictPreDec2Bottleneck().to(local_device)
contract = verify_model_contract(model, cfg.eval_size)
ops = operator_audit(model)
print_setup_summary(cfg)
print_run_header(cfg, paths, train_pairs, val_pairs)
print_model_summary(contract, ops)


## 5. Export and Submission Artifacts


In [ ]:
if not RUN_MODAL_SUBMISSION:
    raise RuntimeError("Set LAB3_SRNET_RUN_MODAL=true to launch the Modal app.")

result = execute_modal_pipeline(cfg)
print_final_summary(result)
artifacts = result["artifact_paths"]
print(
    f"[ARTIFACTS] best={artifacts['best_checkpoint']} "
    f"onnx={artifacts['onnx']} calibration={artifacts['calibration_manifest']} mxq={artifacts['mxq_target']}"
)
